In [3]:
from store.db.db_store import Mongo

mongo = Mongo()

In [7]:
job = mongo.db['t2_job'].find_one(None, {'_id': 0, 'company_id': 1, 'skill_list': 1})
company_id = job.get('company_id')
skill_list = job.get('skill_list')
company_id, skill_list

(ObjectId('64069c833b81c5145c4cdd88'), ['图形', 'GPU图形渲染', 'VTK', '机器学习'])

In [40]:
mongo.db['t2_company'].find_one({'_id': 1234})

In [112]:

mongo.db['t2_company'].find_one({'_id': company_id})
mo = mongo.db['t2_company'].update({'_id': company_id}, {
    '$set': {'skills': ','.join(set(skill_list))}})
mo

/Users/reddyfan/code/proj/RDVS/rdvs/venv/lib/python3.7/site-packages/ipykernel_launcher.py:3: DeprecationWarning: update is deprecated. Use replace_one, update_one or update_many instead.
  This is separate from the ipykernel package so we can avoid doing imports until


{'n': 1, 'nModified': 0, 'ok': 1.0, 'updatedExisting': True}

In [37]:
mongo.db['t2_company'].find_one({'_id': company_id}, {'skills': 1})

{'_id': ObjectId('64069c833b81c5145c4cdd88'), 'skills': '图形,VTK,机器学习,GPU图形渲染'}

In [44]:
from store.db.db_store import get_redis

redis = get_redis()

In [114]:
def add_column_skills():
    skip = int(redis.get('skip:job_welfare'))
    skip = skip if skip else 0
    while True:
        job = mongo.db['t2_job'].find(None, {'_id': 0, 'company_id': 1, 'skill_list': 1}).limit(1).skip(0)[0]
        if job:
            print('数据不存在 或 skill字段已更新完毕！')
            break
        job_skills = job.get('skill_list', [])
        company_skills = mongo.db['t2_company'].find_one({'_id': company_id}, {'skills': 1}).get('skills', [])
        skills = list(set(job_skills + company_skills))
        mongo.db['t2_company'].update_one({'_id': company_id}, {'$set': {'skills': skills}})
        skip += 1
        print('更新到第{skip}条数据')
        redis.set('skip:skills', skip)
    # while True:


In [99]:
add_column_skills()

52


In [111]:
one = slice(0)


{'skill_list': ['图形', 'GPU图形渲染', 'VTK', '机器学习'],
 'company_id': ObjectId('64069c833b81c5145c4cdd88')}